# NovaPay FP&A Analytics: Python Analysis

**Fictional company · monthly data Jan–Jun 2024**

This notebook reads the Excel model (`02_Excel_Model/NovaPay_FPA_Model.xlsx`) and covers:
1. Monthly revenue by stream
2. Revenue growth vs. EBITDA (Jan → Jun bridge)
3. Budget vs. actual (revenue, opex, EBITDA)
4. Expense overruns by category
5. Customer concentration
6. CAC vs. revenue per customer
7. One-month scenario analysis (base / upside / downside)

**Note on forecasting:** six monthly data points are not enough for a statistical forecast. Section 7 is scenario analysis: it applies the assumptions in `Scenario_Inputs` to June 2024 actuals and uses the same logic as the Excel `Scenario_Analysis` sheet, so the two match.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 160)

def money(x):
    return f"-${abs(x):,.0f}" if x < 0 else f"${x:,.0f}"

# Works whether the notebook is run from 04_Python/ or the repo root
candidates = [Path("../02_Excel_Model/NovaPay_FPA_Model.xlsx"), Path("02_Excel_Model/NovaPay_FPA_Model.xlsx")]
excel_file = next(p for p in candidates if p.exists())

revenue   = pd.read_excel(excel_file, sheet_name="Revenue_Data")
expenses  = pd.read_excel(excel_file, sheet_name="Expense_Data")
budget    = pd.read_excel(excel_file, sheet_name="Budget_Data")
kpi       = pd.read_excel(excel_file, sheet_name="KPI_Data")
customers = pd.read_excel(excel_file, sheet_name="Customers")
scenarios = pd.read_excel(excel_file, sheet_name="Scenario_Inputs")
summary   = pd.read_excel(excel_file, sheet_name="Monthly_Summary").sort_values("Month_Number").reset_index(drop=True)

print(f"Loaded {excel_file}")
print(f"Revenue rows: {len(revenue)} | Expense rows: {len(expenses)} | Months: {summary['Month_Number'].nunique()}")

Loaded ../02_Excel_Model/NovaPay_FPA_Model.xlsx
Revenue rows: 22 | Expense rows: 36 | Months: 6


## 1. Monthly revenue by stream

In [2]:
streams = ["Subscription_Revenue", "Transaction_Fee_Revenue", "Premium_Analytics_Revenue", "Setup_Fee_Revenue", "Total_Revenue"]
monthly_revenue = revenue.groupby("Month_Number")[streams].sum()
monthly_revenue.index = summary["Month_Label"].values
monthly_revenue["Recurring_Revenue (excl. setup)"] = monthly_revenue["Total_Revenue"] - monthly_revenue["Setup_Fee_Revenue"]
monthly_revenue

,Subscription_Revenue,Transaction_Fee_Revenue,Premium_Analytics_Revenue,Setup_Fee_Revenue,Total_Revenue,Recurring_Revenue (excl. setup)
Jan-24,1098,1690,199,1200,4187,2987
Feb-24,1098,1755,199,0,3052,3052
Mar-24,1397,2262,199,500,4358,3858
Apr-24,2196,3289,398,700,6583,5883
May-24,2495,3887,398,500,7280,6780
Jun-24,3294,4940,597,700,9531,8831


## 2. Revenue growth vs. EBITDA: January → June bridge

In [3]:
jan, jun = summary.iloc[0], summary.iloc[-1]
bridge = pd.DataFrame({
    "Jan-24": [jan.Actual_Revenue, jan.Gross_Profit, jan.Actual_Expenses, jan.EBITDA],
    "Jun-24": [jun.Actual_Revenue, jun.Gross_Profit, jun.Actual_Expenses, jun.EBITDA],
}, index=["Revenue", "Gross Profit", "Operating Expenses", "EBITDA"])
bridge["Change"] = bridge["Jun-24"] - bridge["Jan-24"]
print(f"Revenue growth Jan -> Jun: {jun.Actual_Revenue / jan.Actual_Revenue - 1:.1%}")
print(f"Gross profit increase: ${bridge.loc['Gross Profit','Change']:,.0f} vs. opex increase: ${bridge.loc['Operating Expenses','Change']:,.0f}")
bridge

Revenue growth Jan -> Jun: 127.6%
Gross profit increase: $3,116 vs. opex increase: $5,900


,Jan-24,Jun-24,Change
Revenue,"4,187.00","9,531.00","5,344.00"
Gross Profit,"2,889.03","6,004.53","3,115.50"
Operating Expenses,"37,850.00","43,750.00","5,900.00"
EBITDA,"-34,960.97","-37,745.47","-2,784.50"


## 3. Budget vs. actual

In [4]:
bva = pd.DataFrame({
    "Month": summary["Month_Label"],
    "Actual_Revenue": summary["Actual_Revenue"],
    "Budget_Revenue": budget.sort_values("Month_Number")["Budget_Total_Revenue"].values,
    "Actual_Opex": summary["Actual_Expenses"],
    "Budget_Opex": budget.sort_values("Month_Number")["Budget_Operating_Expenses"].values,
    "Actual_EBITDA": summary["EBITDA"],
    "Budget_EBITDA": budget.sort_values("Month_Number")["Budget_EBITDA"].values,
})
bva["Revenue_Var"] = bva.Actual_Revenue - bva.Budget_Revenue
bva["Opex_Var (+ = over plan)"] = bva.Actual_Opex - bva.Budget_Opex
bva["EBITDA_Var"] = bva.Actual_EBITDA - bva.Budget_EBITDA

tot = bva.drop(columns="Month").sum()
print(f"Six-month revenue: ${tot.Actual_Revenue:,.0f} vs budget ${tot.Budget_Revenue:,.0f} ({tot.Actual_Revenue / tot.Budget_Revenue - 1:+.1%})")
print(f"Six-month opex:    ${tot.Actual_Opex:,.0f} vs budget ${tot.Budget_Opex:,.0f} ({tot.Actual_Opex / tot.Budget_Opex - 1:+.1%})")
print(f"Six-month EBITDA:  {money(tot.Actual_EBITDA)} vs budget {money(tot.Budget_EBITDA)}")
print(f"Months with opex over plan: {(bva['Opex_Var (+ = over plan)'] > 0).sum()} of 6 | Months with EBITDA below budget: {(bva.EBITDA_Var < 0).sum()} of 6")
bva

Six-month revenue: $34,991 vs budget $29,900 (+17.0%)
Six-month opex:    $245,000 vs budget $234,800 (+4.3%)
Six-month EBITDA:  -$222,217 vs budget -$214,950
Months with opex over plan: 6 of 6 | Months with EBITDA below budget: 6 of 6


,Month,Actual_Revenue,Budget_Revenue,Actual_Opex,Budget_Opex,Actual_EBITDA,Budget_EBITDA,Revenue_Var,Opex_Var (+ = over plan),EBITDA_Var
0,Jan-24,4187,4200,37850,36000,"-34,960.97",-33100,-13,1850,"-1,860.97"
1,Feb-24,3052,3100,38000,36700,"-35,924.64",-34950,-48,1300,-974.64
2,Mar-24,4358,4250,39250,38300,"-36,373.72",-35550,108,950,-823.72
3,Apr-24,6583,5400,41550,40200,"-37,271.05",-36550,1183,1350,-721.05
4,May-24,7280,5950,44600,41500,"-39,940.80",-37500,1330,3100,"-2,440.80"
5,Jun-24,9531,7000,43750,42100,"-37,745.47",-37300,2531,1650,-445.47


## 4. Expense overruns by category

Uses line-item budgets from `Expense_Data`. These total $231.5K, versus $234.8K in `Budget_Data`; see Known limitations in the README.

In [5]:
exp = expenses.assign(Variance=expenses.Actual_Expense - expenses.Budget_Expense)
by_cat = exp.groupby("Expense_Category").agg(
    Actual=("Actual_Expense", "sum"),
    Budget=("Budget_Expense", "sum"),
    Variance=("Variance", "sum"),
    Months_Over_Budget=("Variance", lambda v: int((v > 0).sum())),
).sort_values("Variance", ascending=False)
by_cat["Share_of_Opex"] = by_cat.Actual / by_cat.Actual.sum()
by_cat["Variance_%"] = by_cat.Variance / by_cat.Budget
by_cat

,Actual,Budget,Variance,Months_Over_Budget,Share_of_Opex,Variance_%
Expense_Category,,,,,,
Cloud Hosting,35800,30700,5100,6,0.15,0.17
Marketing Spend,60400,55500,4900,6,0.25,0.09
Professional Fees,9400,8200,1200,2,0.04,0.15
Software,20600,19600,1000,6,0.08,0.05
Payroll,77950,77000,950,5,0.32,0.01
Customer Support Tools,16450,16100,350,5,0.07,0.02
Rent,24400,24400,0,0,0.10,0.00


## 5. Customer concentration

In [6]:
cust = (revenue.groupby("Customer_ID")["Total_Revenue"].sum()
        .rename(index=dict(zip(customers.Customer_ID, customers.Customer_Name)))
        .sort_values(ascending=False).to_frame())
cust["Share"] = cust.Total_Revenue / cust.Total_Revenue.sum()
cust["Cumulative_Share"] = cust.Share.cumsum()
cust

,Total_Revenue,Share,Cumulative_Share
Customer_ID,,,
TechStart Ltd,13929,0.40,0.40
Nova Retail Group,6814,0.19,0.59
Acme Corp,6337,0.18,0.77
BluePeak Systems,3191,0.09,0.87
CloudBridge Inc,2595,0.07,0.94
FinEdge Partners,2125,0.06,1.00


## 6. CAC vs. revenue per customer

In this dataset, CAC = monthly marketing spend ÷ new customers. February has no new customers, so CAC is undefined (shown as blank, not 0).

In [7]:
mkt = expenses[expenses.Expense_Category == "Marketing Spend"].sort_values("Month_Number")["Actual_Expense"].values
k = kpi.sort_values("Month_Number").reset_index(drop=True)
cac = pd.DataFrame({
    "Month": summary["Month_Label"],
    "Marketing_Spend": mkt,
    "New_Customers": k.New_Customers,
    "CAC": np.where(k.New_Customers > 0, mkt / k.New_Customers.replace(0, np.nan), np.nan),
    "Revenue_per_Active_Customer": summary.Actual_Revenue / k.Active_Customers,
})
cac

,Month,Marketing_Spend,New_Customers,CAC,Revenue_per_Active_Customer
0,Jan-24,9200,2,"4,600.00","2,093.50"
1,Feb-24,8700,0,NaN,"1,526.00"
2,Mar-24,10300,1,"10,300.00","1,452.67"
3,Apr-24,9800,1,"9,800.00","1,645.75"
4,May-24,11600,1,"11,600.00","1,456.00"
5,Jun-24,10800,1,"10,800.00","1,906.20"


## 7. One-month scenario analysis (July 2024, from June actuals)

The logic matches the Excel `Scenario_Analysis` sheet:
- Revenue = June revenue × (1 + growth rate)
- EBITDA = June EBITDA + incremental revenue × 65% gross margin − June opex × payroll inflation

These are assumption-driven what-ifs, not predictions.

In [8]:
GM_ASSUMPTION = 0.65
scen = scenarios.set_index("Scenario")[["Customer_Growth_Rate", "Payroll_Inflation"]].copy()
scen["Forecast_Revenue"] = jun.Actual_Revenue * (1 + scen.Customer_Growth_Rate)
scen["Forecast_EBITDA"] = (jun.EBITDA
                           + (scen.Forecast_Revenue - jun.Actual_Revenue) * GM_ASSUMPTION
                           - jun.Actual_Expenses * scen.Payroll_Inflation)
print(f"June actual EBITDA: {money(jun.EBITDA)}")
scen

June actual EBITDA: -$37,745


,Customer_Growth_Rate,Payroll_Inflation,Forecast_Revenue,Forecast_EBITDA
Scenario,,,,
Base Case,0.08,0.04,"10,293.48","-38,999.86"
Upside Case,0.12,0.03,"10,674.72","-38,314.55"
Downside Case,0.04,0.06,"9,912.24","-40,122.66"


## 8. Summary

In [9]:
top_over = by_cat.head(2).index.tolist()
print(f"1. Revenue grew {jun.Actual_Revenue / jan.Actual_Revenue - 1:.0%} Jan->Jun, but EBITDA moved from {money(jan.EBITDA)} to {money(jun.EBITDA)}.")
print(f"2. Revenue beat budget by {tot.Actual_Revenue / tot.Budget_Revenue - 1:.0%} over six months; EBITDA missed budget in {(bva.EBITDA_Var < 0).sum()} of 6 months.")
print(f"3. Largest overruns: {top_over[0]} (${by_cat.Variance.iloc[0]:,.0f}) and {top_over[1]} (${by_cat.Variance.iloc[1]:,.0f}).")
print(f"4. {cust.index[0]} = {cust.Share.iloc[0]:.0%} of six-month revenue; top two = {cust.Cumulative_Share.iloc[1]:.0%}.")
print(f"5. Mar-Jun CAC ranged ${cac.CAC.iloc[2:].min():,.0f}-${cac.CAC.iloc[2:].max():,.0f} per new customer.")
print(f"6. All three scenarios keep EBITDA negative ({money(scen.Forecast_EBITDA.min())} to {money(scen.Forecast_EBITDA.max())}): growth alone does not fix profitability.")

1. Revenue grew 128% Jan->Jun, but EBITDA moved from -$34,961 to -$37,745.
2. Revenue beat budget by 17% over six months; EBITDA missed budget in 6 of 6 months.
3. Largest overruns: Cloud Hosting ($5,100) and Marketing Spend ($4,900).
4. TechStart Ltd = 40% of six-month revenue; top two = 59%.
5. Mar-Jun CAC ranged $9,800-$11,600 per new customer.
6. All three scenarios keep EBITDA negative (-$40,123 to -$38,315): growth alone does not fix profitability.
